# `code/pipeline/p001_43_movers_assignment.py`

Read-only rendering of the script (no outputs; it needs the licensed inputs described in `../../DATA_ACCESS.md`). The .py file is the version of record.


```text
p001_43 — P3: 이동자 — 같은 파트너가 두 회사에서: 구성은 회사가 배정하는가, 파트너 스타일인가

[왜] §6 은 구성–성과 연관을 고정지평 Mundlak 으로 회사 간에 위치시켰다(P001-38 B). 설계 제안 P3 는 같은 질문을 이동 마진에서
 직접 잰다: 한 파트너가 두 회사에서 각 ≥5 딜을 가진 경우, 그녀의 구성 변화가 회사의 (파트너 제외) 구성 변화와 얼마나 함께 움직이나.
 계수 ≈1 이면 구성은 회사 배정, ≈0 이면 파트너 스타일. 파트너 제외(leave-partner-out) 회사 구성이 필수 — 자기 딜을 넣으면 상관을 제조한다.

[구성] sample_v1 NAEU. 스펠 = (파트너, 회사) 의 귀속 딜 ≥5; 스펠 창 = 첫~마지막 딜. 파트너별 스펠 ≥2 → 첫 딜 순으로 전이(연속 스펠 쌍).
 자기 구성 = 스펠 딜의 초기단계 비중 · 섹터 성분(t_cs; exit3 LOO 셀 벤치마크, p001_rescue_common.build) · 단계 성분(t_s).
 회사 LPO 구성 = 스펠 창 안 그 회사의 다른 파트너 딜(≥5 요구) 의 같은 지표.
 D1  Δ자기 = α + β·Δ회사LPO + e (전이 수준; 파트너 군집 부트 400). 비중첩(스펠 창이 겹치지 않는 이동) 과 풀 병기.
 D2  미래 회사 위약: 자기(s1) ← 목적 회사 LPO(s1 창에서 측정) + 출발 회사 LPO(s1). 목적 계수 ≈0 이면 정렬(sorting) 없음.
 D3  위약 이동: 같은 딜수 십분위의 무작위 대체 목적 회사 → Δ회사LPO_placebo; β 붕괴 예상.
 D4  방향별: 후기 지향 이동 vs 초기 지향 이동의 β.
[사전 예측] (2026-09-09, 결과 조회 전)
 이동자(≥5 each) ≈ 370–400, 전이 ≈ 400; 비중첩 비중 ≥ 0.6. D1 β(초기 비중) ∈ [0.4, 0.8], CI 0 배제; t_cs 판 β ∈ [0.3, 0.8].
 D2 목적 회사 계수 ∈ [−0.1, +0.3], CI 0 포함(정렬 미검출). D3 위약 β ∈ [−0.15, +0.15]. D4 비대칭 없음(두 β 차 CI 0 포함).
[판정] D1 β 하한 > 0.3 → GO("구성은 상당 부분 회사 배정"); 상단 < 0.2 → GO("파트너 스타일"); 그 외 PARTIAL. D2 위약 실패(목적 계수 하한 > 0.2)면 verdict 에 표기.

[정정 2026-09-09 — 1차 실행 후] D3 위약(Δ회사_placebo = 대체목적 − 출발) 은 실제 Δ회사와 **출발 회사 항을 공유**해 Δ자기와 기계적 상관을 남긴다
 (1차 결과 β_placebo +0.18*). 목적·출발 회사 수준을 **따로** 넣는 사양으로 교체: D1b Δ자기 ← 목적회사 LPO + 출발회사 LPO;
 D3b Δ자기 ← 대체목적 LPO + 출발회사 LPO. 예측(결과 조회 전): D1b 목적 계수 ∈ [0.4, 0.7], 출발 계수 ∈ [−0.7, −0.4]; D3b 대체목적 계수 ∈ [−0.15, +0.15].
 1차 D1/D2/D4 수치는 재실행에서 불변(seed 동일, 새 회귀는 뒤에 추가).
```


In [ ]:
import numpy as np
import pandas as pd

from p001_rescue_common import build
from p001_v6_common import (EARLY, RESCUE_SHA, V6_SHA, emit, fit, load_sample, log, qci)

rng = np.random.default_rng(20260943)
NB = 400
OUT = {}
d = load_sample()
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))  # gates.py / emit_contract.py sit alongside
from gates import CTX  # noqa: E402
# exit3 for component decomposition
acq = CTX.acq.dropna(subset=["acquiree_uuid", "acquired_on"]).copy(); acq["adt"] = pd.to_datetime(acq["acquired_on"], errors="coerce")
ip = CTX.ipos.dropna(subset=["org_uuid", "went_public_on"]).copy(); ip["idt"] = pd.to_datetime(ip["went_public_on"], errors="coerce")
exit_any = pd.concat([acq.groupby("acquiree_uuid")["adt"].min(), ip.groupby("org_uuid")["idt"].min()], axis=1).min(axis=1)
d["exit_dt"] = d["org_uuid"].map(exit_any)
d["exit3"] = (((d["exit_dt"] - d["dt"]).dt.days <= 1095)).fillna(False).astype(float)
d["y"] = d["year"].astype(str); d["ys"] = d["y"] + "|" + d["stage"]; d["yss"] = d["y"] + "|" + d["cat"] + "|" + d["stage"]
b = build(d, "exit3")
b["early"] = b["stage"].isin(EARLY).astype(float)
b = b[np.isfinite(b["t_cs"])].copy()


In [ ]:
# ── 스펠 ────────────────────────────────────────────────────────────────────
sp = b.groupby(["partner_uuid", "investor_uuid"]).agg(n=("dt", "size"), start=("dt", "min"), end=("dt", "max"), early=("early", "mean"),
                                                        t_cs=("t_cs", "mean"), t_s=("t_s", "mean"), fp=("fp", "first")).reset_index()
sp = sp[sp["n"] >= 5]
cnt = sp.groupby("partner_uuid").size()
movers = cnt.index[cnt >= 2]
sp = sp[sp["partner_uuid"].isin(movers)].sort_values(["partner_uuid", "start"]).reset_index(drop=True)
A = {"n_movers": int(len(movers)), "n_spells": int(len(sp)), "n_female_movers": int(sp.groupby("partner_uuid")["fp"].first().sum())}


def firm_lpo(firm, start, end, partner):
    w = b[(b["investor_uuid"] == firm) & (b["dt"] >= start) & (b["dt"] <= end) & (b["partner_uuid"] != partner)]
    if len(w) < 5:
        return (np.nan, np.nan, np.nan, len(w))
    return (float(w["early"].mean()), float(w["t_cs"].mean()), float(w["t_s"].mean()), len(w))


# 회사 딜 인덱스 가속: 회사별 정렬
rows = []
for p, g in sp.groupby("partner_uuid"):
    g = g.sort_values("start").reset_index(drop=True)
    for i in range(len(g) - 1):
        s1, s2 = g.iloc[i], g.iloc[i + 1]
        f1 = firm_lpo(s1["investor_uuid"], s1["start"], s1["end"], p); f2 = firm_lpo(s2["investor_uuid"], s2["start"], s2["end"], p)
        # 미래 회사 위약: 목적 회사 LPO 를 s1 창에서
        fd = firm_lpo(s2["investor_uuid"], s1["start"], s1["end"], p)
        rows.append({"partner": p, "fp": s1["fp"], "firm1": s1["investor_uuid"], "firm2": s2["investor_uuid"], "overlap": bool(s2["start"] <= s1["end"]),
                     "own_early1": s1["early"], "own_early2": s2["early"], "own_tcs1": s1["t_cs"], "own_tcs2": s2["t_cs"], "own_ts1": s1["t_s"], "own_ts2": s2["t_s"],
                     "firm_early1": f1[0], "firm_early2": f2[0], "firm_tcs1": f1[1], "firm_tcs2": f2[1], "firm_ts1": f1[2], "firm_ts2": f2[2],
                     "dest_early_in_s1": fd[0], "dest_tcs_in_s1": fd[1], "n_col1": f1[3], "n_col2": f2[3]})
T = pd.DataFrame(rows)
for c in ("early", "tcs", "ts"):
    T[f"d_own_{c}"] = T[f"own_{c}2"] - T[f"own_{c}1"]; T[f"d_firm_{c}"] = T[f"firm_{c}2"] - T[f"firm_{c}1"]
T["firm"] = T["partner"]  # boot 군집 = 파트너
A.update({"n_transitions": int(len(T)), "n_nonoverlap": int((~T["overlap"]).sum()), "n_complete_early": int(T.dropna(subset=["d_own_early", "d_firm_early"]).shape[0]),
          "sd_d_own_early": round(float(T["d_own_early"].std()), 4), "sd_d_firm_early": round(float(T["d_firm_early"].std()), 4)})
log(f"[A] 이동자 {A['n_movers']:,} (여성 {A['n_female_movers']}) · 스펠 {A['n_spells']:,} · 전이 {A['n_transitions']:,} · 비중첩 {A['n_nonoverlap']:,} · 완비 {A['n_complete_early']:,} · sd Δ자기 초기비중 {A['sd_d_own_early']:.3f}")
OUT["A_sample"] = A


def creg(df, y, xc, keys, nb=NB):
    dd = df.dropna(subset=[y] + xc).copy()
    if len(dd) < 40:
        return None
    bb = fit(dd, y, xc); grp = {c: g.index.to_numpy() for c, g in dd.groupby("firm")}; kl = list(grp); bs = []
    for _ in range(nb):
        pick = rng.integers(0, len(kl), len(kl)); s = dd.loc[np.concatenate([grp[kl[i]] for i in pick])]
        try: bs.append(fit(s, y, xc))
        except Exception: pass
    bs = np.array(bs); out = {"n": int(len(dd))}
    for k in keys:
        i = xc.index(k); lo, hi = qci(bs[:, i]); se = float(np.std(bs[:, i], ddof=1))
        out[k] = {"coef": round(float(bb[i]), 5), "ci95": [round(lo, 5), round(hi, 5)], "sig": bool(lo > 0 or hi < 0), "se_boot": round(se, 5), "mde80": round(2.8 * se, 4)}
    return out


def show(tag, r, k):
    if not r: log(f"  {tag}: 표본 부족"); return
    log(f"  {tag:<34} β {r[k]['coef']:+.4f} [{r[k]['ci95'][0]:+.3f},{r[k]['ci95'][1]:+.3f}] n={r['n']:,}")


log("\n" + "=" * 100 + "\n[D1] Δ자기 구성 ← Δ회사 LPO 구성 (전이 수준)\n" + "=" * 100)
Tn = T[~T["overlap"]]
D1 = {}
for c in ("early", "tcs", "ts"):
    D1[f"{c}_pooled"] = creg(T, f"d_own_{c}", [f"d_firm_{c}"], [f"d_firm_{c}"]); show(f"D1 {c} 풀", D1[f"{c}_pooled"], f"d_firm_{c}")
    D1[f"{c}_nonoverlap"] = creg(Tn, f"d_own_{c}", [f"d_firm_{c}"], [f"d_firm_{c}"]); show(f"D1 {c} 비중첩", D1[f"{c}_nonoverlap"], f"d_firm_{c}")
OUT["D1_assignment"] = D1
log("\n" + "=" * 100 + "\n[D2] 미래 회사 위약: 자기(s1) ← 목적 회사 LPO(s1 창) + 출발 회사 LPO(s1)\n" + "=" * 100)
D2 = {"early": creg(T, "own_early1", ["dest_early_in_s1", "firm_early1"], ["dest_early_in_s1", "firm_early1"]),
      "tcs": creg(T, "own_tcs1", ["dest_tcs_in_s1", "firm_tcs1"], ["dest_tcs_in_s1", "firm_tcs1"])}
for c, k in (("early", "dest_early_in_s1"), ("tcs", "dest_tcs_in_s1")):
    if D2[c]: log(f"  D2 {c:<6} 목적 {D2[c][k]['coef']:+.4f} [{D2[c][k]['ci95'][0]:+.3f},{D2[c][k]['ci95'][1]:+.3f}] · 출발 {D2[c]['firm_' + c + '1']['coef']:+.4f} n={D2[c]['n']:,}")
OUT["D2_future_firm_placebo"] = D2
log("\n" + "=" * 100 + "\n[D3] 위약 이동: 같은 딜수 십분위의 무작위 대체 목적 회사\n" + "=" * 100)
fsize = b.groupby("investor_uuid").size()
fdec = pd.qcut(fsize.rank(method="first"), 10, labels=False)
pool = {k: v.index.to_numpy() for k, v in fsize.groupby(fdec)}
pl = []
for _, r in T.iterrows():
    dec = fdec.get(r["firm2"]); cand = pool.get(dec, np.array([]))
    cand = cand[cand != r["firm2"]]
    if len(cand) == 0: pl.append(np.nan); continue
    f_alt = str(rng.choice(cand))
    # 대체 목적 회사의 LPO 초기비중 (s2 창, 파트너 제외 — 그 회사엔 파트너 딜이 없으므로 그대로)
    w = b[(b["investor_uuid"] == f_alt) & (b["dt"] >= sp.loc[(sp.partner_uuid == r["partner"]) & (sp.investor_uuid == r["firm2"]), "start"].iloc[0]) &
          (b["dt"] <= sp.loc[(sp.partner_uuid == r["partner"]) & (sp.investor_uuid == r["firm2"]), "end"].iloc[0])]
    pl.append(float(w["early"].mean()) if len(w) >= 5 else np.nan)
T["firm_early2_placebo"] = pl; T["d_firm_early_placebo"] = T["firm_early2_placebo"] - T["firm_early1"]
D3 = creg(T, "d_own_early", ["d_firm_early_placebo"], ["d_firm_early_placebo"]); show("D3 위약 이동 β (차분; 출발 항 공유 — 참고)", D3, "d_firm_early_placebo")
OUT["D3_placebo_mover"] = D3
log("\n[D1b/D3b] 목적·출발 회사 수준을 따로: Δ자기 ← 목적 LPO + 출발 LPO / 대체목적 LPO + 출발 LPO")
D1b = creg(T, "d_own_early", ["firm_early2", "firm_early1"], ["firm_early2", "firm_early1"])
D3b = creg(T, "d_own_early", ["firm_early2_placebo", "firm_early1"], ["firm_early2_placebo", "firm_early1"])
if D1b: log(f"  D1b 목적 {D1b['firm_early2']['coef']:+.4f} [{D1b['firm_early2']['ci95'][0]:+.3f},{D1b['firm_early2']['ci95'][1]:+.3f}] · 출발 {D1b['firm_early1']['coef']:+.4f} n={D1b['n']:,}")
if D3b: log(f"  D3b 대체목적 {D3b['firm_early2_placebo']['coef']:+.4f} [{D3b['firm_early2_placebo']['ci95'][0]:+.3f},{D3b['firm_early2_placebo']['ci95'][1]:+.3f}] · 출발 {D3b['firm_early1']['coef']:+.4f} n={D3b['n']:,}")
OUT["D1b_levels"] = D1b; OUT["D3b_placebo_levels"] = D3b
log("\n" + "=" * 100 + "\n[D4] 방향별: 목적 회사가 더 초기 지향 vs 더 후기 지향\n" + "=" * 100)
up = T[T["d_firm_early"] < 0]; dn_ = T[T["d_firm_early"] > 0]
D4 = {"to_later_stage_firm": creg(up, "d_own_early", ["d_firm_early"], ["d_firm_early"]), "to_earlier_stage_firm": creg(dn_, "d_own_early", ["d_firm_early"], ["d_firm_early"])}
for k, v in D4.items(): show(f"D4 {k}", v, "d_firm_early")
OUT["D4_direction"] = D4


In [ ]:
# ── 판정 ────────────────────────────────────────────────────────────────────
bE = D1["early_nonoverlap"]["d_firm_early"] if D1["early_nonoverlap"] else D1["early_pooled"]["d_firm_early"]
if bE["ci95"][0] > 0.3:
    status, call = "GO", "이동 시 파트너의 단계 구성이 회사 구성과 함께 움직인다 — 구성은 상당 부분 회사 배정"
elif bE["ci95"][1] < 0.2:
    status, call = "GO", "이동해도 파트너 구성은 회사 구성을 따르지 않는다 — 파트너 스타일"
else:
    status, call = "PARTIAL", "배정 비중 결정 불가(구간 넓음)"
d2e = D2["early"]["dest_early_in_s1"] if D2["early"] else None
sorting_flag = bool(d2e and d2e["ci95"][0] > 0.2)
pred = {"A_movers_370_400": 370 <= A["n_movers"] <= 400, "A_nonoverlap_share_ge_0.6": A["n_nonoverlap"] / max(A["n_transitions"], 1) >= 0.6,
        "D1_early_beta_in_[0.4,0.8]_sig": bool(0.4 <= bE["coef"] <= 0.8 and bE["sig"]),
        "D1_tcs_beta_in_[0.3,0.8]": bool(D1["tcs_nonoverlap"] and 0.3 <= D1["tcs_nonoverlap"]["d_firm_tcs"]["coef"] <= 0.8),
        "D2_dest_in_[-0.1,0.3]_incl0": bool(d2e and -0.1 <= d2e["coef"] <= 0.3 and not d2e["sig"]),
        "D3_placebo_in_pm0.15": bool(D3 and abs(D3["d_firm_early_placebo"]["coef"]) <= 0.15),
        "D1b_dest_in_[0.4,0.7]": bool(D1b and 0.4 <= D1b["firm_early2"]["coef"] <= 0.7),
        "D3b_alt_in_pm0.15": bool(D3b and abs(D3b["firm_early2_placebo"]["coef"]) <= 0.15),
        "D4_no_asymmetry": bool(D4["to_later_stage_firm"] and D4["to_earlier_stage_firm"] and abs(D4["to_later_stage_firm"]["d_firm_early"]["coef"] - D4["to_earlier_stage_firm"]["d_firm_early"]["coef"]) < 0.3)}
pred = {k: bool(v) for k, v in pred.items()}
OUT["prediction_check"] = pred
OUT["flags"] = {"sorting_detected": sorting_flag}
verdict = (f"이동자 {A['n_movers']} · 전이 {A['n_transitions']} (비중첩 {A['n_nonoverlap']}) | D1 초기비중 β 비중첩 {bE['coef']:+.3f} [{bE['ci95'][0]:+.3f},{bE['ci95'][1]:+.3f}] · 풀 {D1['early_pooled']['d_firm_early']['coef']:+.3f} · "
           f"t_cs β {D1['tcs_nonoverlap']['d_firm_tcs']['coef'] if D1['tcs_nonoverlap'] else float('nan'):+.3f} | D2 목적 회사 위약 {d2e['coef'] if d2e else float('nan'):+.3f} [{d2e['ci95'][0] if d2e else float('nan'):+.3f},{d2e['ci95'][1] if d2e else float('nan'):+.3f}] | "
           f"D3 위약(차분) β {D3['d_firm_early_placebo']['coef'] if D3 else float('nan'):+.3f} · D1b 목적 {D1b['firm_early2']['coef'] if D1b else float('nan'):+.3f} / D3b 대체목적 {D3b['firm_early2_placebo']['coef'] if D3b else float('nan'):+.3f} "
           f"[{D3b['firm_early2_placebo']['ci95'][0] if D3b else float('nan'):+.3f},{D3b['firm_early2_placebo']['ci95'][1] if D3b else float('nan'):+.3f}] | D4 후기행 {D4['to_later_stage_firm']['d_firm_early']['coef'] if D4['to_later_stage_firm'] else float('nan'):+.3f} / 초기행 {D4['to_earlier_stage_firm']['d_firm_early']['coef'] if D4['to_earlier_stage_firm'] else float('nan'):+.3f} — "
           f"{call}{' · 정렬 검출(D2)' if sorting_flag else ''} (예측 적중 {sum(pred.values())}/{len(pred)})")
emit("P001-43", "P3: 이동자 — 파트너 구성 변화 ← 회사(파트너 제외) 구성 변화; 미래 회사·위약 이동·방향 위약", status, OUT,
     prediction="이동자 370–400; 비중첩≥60%; D1 β∈[0.4,0.8]*; t_cs β∈[0.3,0.8]; D2 목적 ∈[−0.1,0.3] ns; D3 |β|≤0.15; D4 비대칭 없음",
     verdict=verdict, kill_met=False, n=int(A["n_transitions"]),
     extra={"stage": 7, "feeds": "v6 설계 제안 P3 → §6 위치 질문의 직접 추정", "slug": "movers_assignment", "builds_on": "P001-38", "common_sha256_16": RESCUE_SHA, "v6_common_sha256_16": V6_SHA})
log("done")
